In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).view(-1, 1)

print(X_train_t.shape, y_train_t.shape)
print(X_test_t.shape, y_test_t.shape)



In [ ]:
# 2. Create TensorDataset objects


from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

len(train_dataset), len(test_dataset)


In [ ]:
# 3. Create DataLoaders

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)




In [ ]:
# 4. Print shape of one batch
xb, yb = next(iter(train_loader))
print("X batch:", xb.shape)
print("y batch:", yb.shape)



In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

xb, yb = next(iter(train_loader))

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    img = xb[i].permute(1, 2, 0).numpy()  # (H, W, C)
    axes[i].imshow(img)
    axes[i].set_title(f"Age: {yb[i].item():.0f}")
    axes[i].axis("off")
plt.show()



In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class AgeRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        # Flatten 3x36x36 into a vector
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(3*36*36, 512),   # 1
            nn.ReLU(),
            nn.Linear(512, 128),       # 2
            nn.ReLU(),
            nn.Linear(128, 32),        # 3
            nn.ReLU(),
            nn.Linear(32, 1)           # 4

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)


In [ ]:
# Task 2: Write your training loop here:
import torch

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
# Task 4: Define device, model, loss, optimizer:
import torch.optim as optim
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AgeRegressor().to(device)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

device


In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []

for epoch in range(20):
    tr_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    va_loss = validate(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch+1:02d} | train loss: {tr_loss:.4f} | val loss: {va_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.title("Loss over epochs")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)

with torch.no_grad():
    preds = model(xb).squeeze(1)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    img = xb[i].cpu().permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"P:{preds[i].item():.1f}\nT:{yb[i].item():.0f}")
    axes[i].axis("off")
plt.show()
